In [4]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

print("--- 1. Data Understanding ---")
df = pd.read_csv('Top 1000 IMDB movies (1).csv')
df = df.dropna(subset=['Description'])
df['Target_Class'] = (df['Movie Rating'] >= 8.0).astype(int)

print(f"Total valid samples: {len(df)}")
print("Class Distribution:\n", df['Target_Class'].value_counts(normalize=True))
print("\nSample Description:\n", df['Description'].iloc[0])

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return " ".join(cleaned_tokens)

print("\n--- 2. Applying NLP Preprocessing ---")
df['Cleaned_Description'] = df['Description'].apply(clean_text)

X_train, X_test, y_train, y_test = train_test_split(
    df['Cleaned_Description'], df['Target_Class'], test_size=0.2, random_state=42, stratify=df['Target_Class']
)

print("\n--- 3. Feature Engineering ---")

bow_vectorizer = CountVectorizer(max_features=2000)
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

tfidf_vectorizer = TfidfVectorizer(max_features=2000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)



print("\n--- 4 & 5. Model Building and Evaluation ---")

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier(random_state=42)
}

vectorizations = {
    "Bag of Words": (X_train_bow, X_test_bow),
    "TF-IDF": (X_train_tfidf, X_test_tfidf)
}

results = []

for vec_name, (X_tr, X_te) in vectorizations.items():
    print(f"\nEvaluating with {vec_name}:")
    for model_name, model in models.items():
        model.fit(X_tr, y_train)
        y_pred = model.predict(X_te)
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        
        results.append({
            'Vectorization': vec_name,
            'Model': model_name,
            'Accuracy': acc,
            'Precision': prec,
            'Recall': rec,
            'F1 Score': f1
        })
        print(f"  {model_name} -> Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f}")

results_df = pd.DataFrame(results)

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/shivamsalunke/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/shivamsalunke/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/shivamsalunke/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/shivamsalunke/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


--- 1. Data Understanding ---
Total valid samples: 1000
Class Distribution:
 Target_Class
0    0.537
1    0.463
Name: proportion, dtype: float64

Sample Description:
 Two imprisoned men bond over a number of years, finding solace and eventual redemption through acts of common decency.

--- 2. Applying NLP Preprocessing ---

--- 3. Feature Engineering ---

--- 4 & 5. Model Building and Evaluation ---

Evaluating with Bag of Words:
  Logistic Regression -> Acc: 0.4900 | Prec: 0.4526 | Rec: 0.4624 | F1: 0.4574
  Naive Bayes -> Acc: 0.4950 | Prec: 0.4535 | Rec: 0.4194 | F1: 0.4358
  Decision Tree -> Acc: 0.5150 | Prec: 0.4773 | Rec: 0.4516 | F1: 0.4641

Evaluating with TF-IDF:
  Logistic Regression -> Acc: 0.4750 | Prec: 0.4000 | Rec: 0.2581 | F1: 0.3137
  Naive Bayes -> Acc: 0.5100 | Prec: 0.4510 | Rec: 0.2473 | F1: 0.3194
  Decision Tree -> Acc: 0.5050 | Prec: 0.4643 | Rec: 0.4194 | F1: 0.4407
